In [ ]:
import numpy as np
import time
N = 1024
A_cpu = np.random.rand(N, N).astype(np.float32)
B_cpu = np.random.rand(N, N).astype(np.float32)
start = time.time()
C_cpu = np.matmul(A_cpu, B_cpu)
end = time.time()
print(f"Mnożenie macierzy na CPU trwało: {end - start:.5f} sekund.")


Mnożenie macierzy na CPU trwało: 0.02277 sekund.


In [ ]:
!nvidia-smi

Wed Oct 21 12:11:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import cupy as cp
import time
N = 1024
A_cpu = np.random.rand(N,N).astype(np.float32)
B_cpu = np.random.rand(N,N).astype(np.float32)
A_gpu = cp.array(A_cpu)
B_gpu = cp.array(B_cpu)
start = time.time()
C_gpu = cp.matmul(A_gpu, B_gpu)
cp.cuda.Stream.null.synchronize()  # Synchronizacja z GPU
end = time.time()
print(f"Mnożenie macierzy na GPU trwało: {end - start:.5f} sekund.")

del A_gpu
del B_gpu
del C_gpu


CUBLASError: CUBLAS_STATUS_EXECUTION_FAILED

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!pip install pycuda

In [ ]:
kernel_code = """
__global__ void matrixMul(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    float sum = 0.0;
    if(row < N && col < N) {
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}
"""


In [ ]:
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

mod = SourceModule(kernel_code)

matrixMul = mod.get_function("matrixMul")

N = 1024
A_cpu = np.random.rand(N,N).astype(np.float32)
B_cpu = np.random.rand(N,N).astype(np.float32)


C_cpu = np.empty((N, N), dtype=np.float32)

A_gpu = cuda.mem_alloc(A_cpu.nbytes)
B_gpu = cuda.mem_alloc(B_cpu.nbytes)
C_gpu = cuda.mem_alloc(C_cpu.nbytes)

cuda.memcpy_htod(A_gpu, A_cpu)
cuda.memcpy_htod(B_gpu,B_cpu)

block_size = (32, 32, 1)
grid_size = (int(N/32), int(N/32), 1)

matrixMul(A_gpu, B_gpu, C_gpu, np.int32(N), block=block_size, grid=grid_size)
start = time.time()
cuda.Context.synchronize()
cuda.memcpy_dtoh(C_cpu, C_gpu)
end = time.time()


print(f"Mnożenie macierzy na CPU trwało: {end - start:.5f} sekund.")
del A_gpu
del B_gpu
del C_gpu


Mnożenie macierzy na CPU trwało: 0.00759 sekund.


In [ ]:
import matplotlib.pyplot as plt

CPU_Times = []
CUDA_Times = []
Kernel_times = []

N_table = [128,256,512,1024]
for i in N_table:
  np.random.seed(0)
  N = i
  A_cpu = np.random.rand(N, N).astype(np.float32)
  B_cpu = np.random.rand(N, N).astype(np.float32)

  start = time.time()
  C_cpu = np.matmul(A_cpu, B_cpu)
  end = time.time()

  CPU_Times.append(end-start)

for i in N_table:
  np.random.seed(0)
  N = i
  A_cpu = np.random.rand(N, N).astype(np.float32)
  B_cpu = np.random.rand(N, N).astype(np.float32)

  A_gpu = cp.array(A_cpu)
  B_gpu = cp.array(B_cpu)
  start = time.time()
  C_gpu = cp.matmul(A_gpu, B_gpu)
  cp.cuda.Stream.null.synchronize()  # Synchronizacja z GPU
  end = time.time()

  CUDA_Times.append(end-start)

  del A_gpu
  del B_gpu
  del C_gpu


for i in N_table:
  np.random.seed(0)
  N = i
  A_cpu = np.random.rand(N, N).astype(np.float32)
  B_cpu = np.random.rand(N, N).astype(np.float32)
  C_cpu = np.empty((N, N), dtype=np.float32)

  A_gpu = cuda.mem_alloc(A_cpu.nbytes)
  B_gpu = cuda.mem_alloc(B_cpu.nbytes)
  C_gpu = cuda.mem_alloc(C_cpu.nbytes)

  cuda.memcpy_htod(A_gpu, A_cpu)
  cuda.memcpy_htod(B_gpu,B_cpu)

  block_size = (32, 32, 1)
  grid_size = (int(N/32), int(N/32), 1)

  matrixMul(A_gpu, B_gpu, C_gpu, np.int32(N), block=block_size, grid=grid_size)
  start = time.time()
  cuda.Context.synchronize()
  cuda.memcpy_dtoh(C_cpu, C_gpu)
  end = time.time()

  Kernel_times.append(end-start)

  del A_gpu
  del B_gpu
  del C_gpu





CUBLASError: CUBLAS_STATUS_EXECUTION_FAILED